# Cascaded Deep Spatial Refinement: Methodology for Sub-2-Meter Satellite Mapping (10m -> 2.5m -> 1.25m)
## Theoretical Formulation, Residual Dense Architecture, Aerial Ground Truth Guidance, and Multi-Scale Edge Evaluation

**Department of Computer Science & Engineering | Graphic Era Deemed to be University, Dehradun**  
**Project Team ID:** MP2026CSE170  
**Faculty Guide:** Mr. Vishnu Singh Shekhawat  
**Students:** Aaditya Singh Rawat, Anmol Rawat, Dhriti Rana, Ayush Singh Rauthan  

---

## 1. Executive Summary & Research Motivation

### 1.1 The Resolution Gap in Satellite Remote Sensing
Standard 4x super-resolution models (such as baseline SEN2SR) upscale Sentinel-2 visible bands from **10m to 2.5m** ground sampling distance (GSD). While 2.5m provides clear terrain and broad land-use differentiation, critical physical structures remain ambiguous:
- Individual rural buildings and urban roof footprints typically measure 8-15m across, spanning only 3-5 pixels at 2.5m.
- Secondary roads and canal corridors (3-6m wide) remain 1-2 pixel blur lines.
- Cadastral plot divisions and field access tracks suffer from edge softness.

### 1.2 Why Not a Direct 10x Single-Step Leap (10m -> 1m)?
Attempting to train a neural network to jump directly from 10m to 1m in a single step is mathematically ill-posed and unstable:
- **Pixel Explosion Factor:** A single 10m input pixel must simultaneously predict 10 x 10 = 100 high-resolution sub-pixels.
- **Under-Constrained Optimization:** The space of plausible 100-pixel configurations for an ambiguous low-resolution pixel is vast, leading to mode averaging (severe blurring) or training divergence.
- **Spectral Instability:** Forcing a network to simultaneously resolve 10 spectral channels while scaling 10x causes severe radiometric drift.

### 1.3 The Cascaded Solution
To overcome this fundamental limitation, this project introduces a **Two-Stage Cascaded Framework** that decomposes the problem into two well-conditioned sub-tasks:

```text
Raw Sentinel-2 (10m, 10 Bands)
           |
           v  Stage 1: 4x Multispectral Synthesis (SEN2SR + Compound Loss)
Intermediate Output (2.5m, 10 Bands with Calibrated Radiometry & Indices)
           |
           v  Stage 2: 2x Spatial Contour Refinement (Residual Dense Network)
Ultra-Resolution Output (1.25m GSD, Sharp Geometric Contours)
```

## 2. Stage 2 Refiner Architecture: Residual Dense Network (RDN)

The Stage 2 spatial refiner focuses strictly on high-frequency geometric contours (building edges, road boundaries, waterlines). To achieve this without losing low-level spatial memory, we implement a **Residual Dense Network (RDN)**.

### 2.1 Key Architectural Mechanisms
1. **Contiguous Memory Mechanism:** Every convolutional layer inside each Residual Dense Block (RDB) receives concatenated feature maps from all preceding layers within that block. This ensures low-level edge features flow directly to the upsampler without vanishing.
2. **Local Residual Learning (LRL):** Bypasses the dense block via a residual identity connection, stabilizing gradient propagation.
3. **Global Feature Fusion (GFF):** Combines features across all 5 dense blocks with a 1x1 bottleneck convolution, capturing multi-receptive field representations.
4. **Sub-Pixel Convolution (PixelShuffle):** Upscales features by 2x using periodic shuffling ($r^2 C \to C$), completely avoiding transposed convolution checkerboard artifacts.

In [ ]:
import torch
import torch.nn as nn

class ResidualDenseBlock(nn.Module):
    """Residual Dense Block (RDB) with dense feature concatenation and local residual learning."""
    def __init__(self, in_channels=64, growth_rate=32, num_layers=4):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels + i * growth_rate, growth_rate, kernel_size=3, padding=1),
                nn.LeakyReLU(0.2, inplace=True)
            )
            for i in range(num_layers)
        ])
        # 1x1 Local Feature Fusion (LFF)
        self.lff = nn.Conv2d(in_channels + num_layers * growth_rate, in_channels, kernel_size=1)

    def forward(self, x):
        features = [x]
        for layer in self.layers:
            out = layer(torch.cat(features, dim=1))
            features.append(out)
        # Local residual connection
        return x + self.lff(torch.cat(features, dim=1))


class DeepSpatialRefiner2x(nn.Module):
    """Stage 2 Cascaded 2x Refiner: maps 2.5m intermediate features to 1.25m ultra-res."""
    def __init__(self, in_channels=3, out_channels=3, num_features=64, num_blocks=5, growth_rate=32):
        super().__init__()
        # Shallow feature extraction
        self.conv_in = nn.Conv2d(in_channels, num_features, kernel_size=3, padding=1)
        
        # 5 Residual Dense Blocks
        self.rdb_blocks = nn.ModuleList([
            ResidualDenseBlock(in_channels=num_features, growth_rate=growth_rate, num_layers=4)
            for _ in range(num_blocks)
        ])
        
        # Global Feature Fusion (GFF)
        self.gff = nn.Sequential(
            nn.Conv2d(num_features * num_blocks, num_features, kernel_size=1),
            nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)
        )
        
        # 2x Sub-Pixel Upsampler (PixelShuffle)
        self.upsampler = nn.Sequential(
            nn.Conv2d(num_features, num_features * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(upscale_factor=2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(num_features, out_channels, kernel_size=3, padding=1)
        )

    def forward(self, x):
        f0 = self.conv_in(x)
        block_outs = []
        feat = f0
        for rdb in self.rdb_blocks:
            feat = rdb(feat)
            block_outs.append(feat)
        
        # Global feature fusion + global residual connection
        f_fused = self.gff(torch.cat(block_outs, dim=1)) + f0
        
        # Sub-pixel upscaling
        out = self.upsampler(f_fused)
        return torch.clamp(out, 0.0, 1.0)

# Instantiate model and print parameter summary
model = DeepSpatialRefiner2x()
total_params = sum(p.numel() for p in model.parameters())
print(f"DeepSpatialRefiner2x Parameters: {total_params:,}")


## 3. Training Strategy & Optimization Loss

### 3.1 Frozen Stage 1 Backbone
During Stage 2 training, the fine-tuned Stage 1 SEN2SR Lite generator is **completely frozen**. This serves two critical purposes:
1. **Prevents Catastrophic Forgetting:** The 10-band radiometric calibration and spectral index preservation achieved in Stage 1 remain intact.
2. **Accelerates Optimization:** Intermediate 2.5m tensors can be pre-computed and cached in RAM, allowing rapid 30-epoch training in under 12 minutes on a standard GPU.

### 3.2 High-Resolution Ground Truth Supervision
The Stage 2 refiner is trained on paired Sentinel-2 and native **~1m National Agriculture Imagery Program (NAIP)** airborne photography. The loss function balances absolute pixel fidelity with high-frequency spatial gradients:

$$\mathcal{L}_{\text{refiner}} = \mathcal{L}_1(I_{\text{ultra}}, I_{\text{NAIP}}) + \lambda_{\text{edge}} \mathcal{L}_{\text{Sobel}}(I_{\text{ultra}}, I_{\text{NAIP}})$$

where $\lambda_{\text{edge}} = 0.5$, penalizing blurred structural contours without introducing artificial hallucinations.

## 4. Seamless Inference: 2D Hann Cosine Window Blending

When super-resolving large geographic scenes, standard tiled inference causes visible grid seams and boundary jitter. To solve this, Stage 2 employs an overlapping patch inference mechanism weighted by a **2D Hann Cosine Window**:

$$w(x, y) = \sin^2\left(\frac{\pi x}{W - 1}\right) \cdot \sin^2\left(\frac{\pi y}{H - 1}\right)$$

Patches are processed with a 50% spatial overlap. Across overlapping areas, predictions are accumulated and normalized by the sum of window weights. Because the Hann window smoothly tapers to zero at tile borders, seam lines and boundary artifacts are completely eliminated.

## 5. Quantitative Acuity Evaluation & Multi-Scale Results

Evaluation was conducted on uncompressed 16-bit analytical Sentinel-2 tiles using the Tenengrad spatial gradient edge acuity metric (mean $\|\nabla I\|$):

| Resolution Tier | Ground Sampling Distance (GSD) | RGB Edge Acuity | Sharpness Gain vs Native 10m |
| :--- | :---: | :---: | :---: |
| **1. Native Input** | 10.0 meters | 0.06594 | 0.00% (Reference) |
| **2. Pretrained Baseline SEN2SR** | 2.5 meters | 0.07737 | +17.33% |
| **3. Fine-Tuned Stage 1 (Proposed)** | 2.5 meters | 0.09009 | +36.64% (2x Baseline) |
| **4. Cascaded Stage 2 Ultra-Res (Proposed)** | **1.25 meters** | **0.12336** | **+87.08%** (5x Baseline) |

### Key Visual Takeaways
- **Building Footprints:** Blurry clusters at 10m resolve into distinct geometric structures at 1.25m.
- **Water-Land Interfaces:** Shorelines and canal edges show continuous delineation without pixel staircasing.
- **Agricultural Plots:** Field boundaries and farm tracks exhibit clear spatial definition.

## 6. Execution and Replication

All experiments and models described in this methodology are implemented in the following notebooks:
- **Training Notebook:** `notebooks/05_ultra_highres_1m_experiments/Train_1Meter_Cascaded_Refiner.ipynb`
- **Evaluation Notebook:** `notebooks/05_ultra_highres_1m_experiments/SEN2SR_1Meter_4Way_Comparison.ipynb`
- **Model Checkpoint:** Saved to `Results/sen2sr_cascaded_1meter_refiner.pth`